# Introduction to Vector Stores


- Use Langchain documentation to create vector stores using chromadb, milvus, weaviate, pinecone: https://python.langchain.com/docs/integrations/vectorstores/
- Create embeddings of a document and index into vector stores: https://huggingface.co/blog/getting-started-with-embeddings
- For the task you will be indexing the book: https://www.planetebook.com/free-ebooks/crime-and-punishment.pdf
- Experiment different vector stores and other variables for and present your findings for optimizing retrieval (the most appropriate parts should be returned for any query)
- Hint: chunking

**Steps**
- Go through the given readings. Take an hour at max.
- Index document as vectors. Use any documentation to help you, but avoid using AI Tools.
- For embedding model, use an open source model from huggingface.
- Query should return most appropriate parts in relevance to the query. *Remember, your task is to build effective retrieval*


**Resources for understanding vector search**
- https://weaviate.io/blog/vector-search-explained


**Additional resources for understanding embeddings**
- https://cohere.com/llmu/text-embeddings
- https://docs.cohere.com/v2/docs/embeddings
- https://docs.cohere.com/v2/docs/playground-overview

# Week 8 – Day 2: Introduction to Vector Stores

## Objective

In this notebook we will complete the assignment from the original notebook:

> Build an effective retrieval system for **Crime and Punishment** by creating embeddings, indexing document chunks in a vector store, and testing different retrieval settings.

The main pipeline is:

**PDF → Text → Chunks → Embeddings → Vector Store → Query → Relevant Passages**

The most important part of this exercise is **retrieval quality**. We are not building a full chatbot yet; we are building the retrieval component that can find the most appropriate passages for a question.

### What we will learn

1. What embeddings are
2. Why documents must be chunked
3. How Hugging Face embeddings work
4. How ChromaDB stores vectors
5. How semantic similarity search works
6. How to inspect retrieved passages
7. How chunk size, overlap, and `k` affect retrieval
8. How to compare retrieval configurations
9. How to write a conclusion based on experimental evidence

## 1. Original assignment requirements

The supplied notebook asks us to:

- Use LangChain documentation for vector stores such as ChromaDB, Milvus, Weaviate and Pinecone.
- Create embeddings of a document and index them into a vector store.
- Use an **open-source Hugging Face embedding model**.
- Index the supplied **Crime and Punishment** PDF.
- Experiment with vector stores and other variables to optimize retrieval.
- Remember that the task is to build **effective retrieval**.
- The notebook specifically gives **chunking** as a hint.

We will therefore use **ChromaDB as the main local vector store**, because it is straightforward to run in Google Colab, and we will perform systematic experiments with chunk size, overlap, and number of retrieved chunks.

> Note: Cloud vector stores such as Pinecone may require an API key, while Milvus/Weaviate may require additional services. The core assignment can be completed locally with ChromaDB.

## 2. Install the required packages

Run this cell first in Google Colab.

We install:

- `langchain` – framework components
- `langchain-community` – document loaders and integrations
- `langchain-text-splitters` – document chunking
- `langchain-chroma` – ChromaDB integration
- `langchain-huggingface` – Hugging Face embedding integration
- `pypdf` – PDF reading
- `sentence-transformers` – open-source embedding models
- `chromadb` – vector database

If Colab asks for a runtime restart after installation, restart it and then continue from the next cell.

In [1]:
!pip install -q -U langchain langchain-community langchain-text-splitters langchain-chroma langchain-huggingface pypdf sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2

## 3. Import libraries

These imports give us the tools needed for the complete pipeline.

In [2]:
import os
import re
import shutil
import requests
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Libraries imported successfully.")

/tmp/ipykernel_2019/3929156681.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Libraries imported successfully.


## 4. Download the book

The assignment specifies the Planet eBook version of **Crime and Punishment**.

We download the PDF directly into the Colab working directory.

If you already have the PDF, you can upload it to Colab instead and set `PDF_PATH` to the uploaded filename.

In [3]:
PDF_URL = "https://www.planetebook.com/free-ebooks/crime-and-punishment.pdf"
PDF_PATH = "crime-and-punishment.pdf"

if not os.path.exists(PDF_PATH):
    response = requests.get(PDF_URL, timeout=60)
    response.raise_for_status()
    with open(PDF_PATH, "wb") as f:
        f.write(response.content)

print(f"PDF ready: {PDF_PATH}")
print(f"File size: {os.path.getsize(PDF_PATH) / (1024*1024):.2f} MB")

PDF ready: crime-and-punishment.pdf
File size: 2.37 MB


## 5. Load the PDF

A PDF is not yet a vector database. First we extract its text.

`PyPDFLoader` reads the PDF page by page. Each returned document normally contains:

- `page_content` → the text
- `metadata` → information such as the page number and source

This page-level structure is useful because later we can tell the user where a retrieved passage came from.

In [4]:
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print("Number of pages loaded:", len(pages))
print("\nFirst page metadata:")
print(pages[0].metadata)

print("\nFirst page preview:")
print(pages[0].page_content[:1500])

Number of pages loaded: 767

First page metadata:
{'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0)', 'creationdate': '2008-02-06T20:52:29+11:00', 'subject': 'Download classic literature as completely free eBooks from Planet eBook.', 'author': 'Fyodor Dostoevsky', 'moddate': '2008-07-06T19:05:06+10:00', 'title': 'Crime and Punishment', 'trapped': '/False', 'source': 'crime-and-punishment.pdf', 'total_pages': 767, 'page': 0, 'page_label': '1'}

First page preview:
Download free eBooks of classic literature, books and 
novels at Planet eBook. Subscribe to our free eBooks blog 
and email newsletter.
Crime and Punishment
By Fyodor Dostoevsky


## 6. Understand why chunking is necessary

We should **not** normally create one embedding for the entire book.

Instead, we divide the book into smaller pieces called **chunks**.

### Why?

Suppose a user asks:

> Why did Raskolnikov feel guilty?

The answer may be in a few paragraphs. If we embedded the entire book as one vector, the retrieval system would have no way to identify that small relevant section.

With chunking:

**Book → hundreds/thousands of chunks → one embedding per chunk**

This gives the vector store much finer retrieval resolution.

### Chunk size

`chunk_size=500` means that each chunk is approximately 500 characters, subject to the splitter's boundary rules.

### Chunk overlap

`chunk_overlap=50` means adjacent chunks share approximately 50 characters.

Overlap helps prevent an important sentence from being split exactly at a boundary.

In [5]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(pages)

print("Pages:", len(pages))
print("Chunks:", len(chunks))
print("Average chunk length:", round(sum(len(c.page_content) for c in chunks) / len(chunks), 1))

print("\nExample chunk:")
print(chunks[0].page_content[:1000])

Pages: 767
Chunks: 2856
Average chunk length: 412.1

Example chunk:
Download free eBooks of classic literature, books and 
novels at Planet eBook. Subscribe to our free eBooks blog 
and email newsletter.
Crime and Punishment
By Fyodor Dostoevsky


## 7. Create Hugging Face embeddings

The assignment requires an **open-source model from Hugging Face**.

We will use:

`sentence-transformers/all-MiniLM-L6-v2`

This model converts text into numerical vectors.

For example:

```text
"Raskolnikov felt guilty"
        ↓
[0.12, -0.08, 0.31, ...]
```

The actual vector contains many dimensions. We do not interpret individual numbers directly. Instead, the vector represents semantic information that can be compared with other vectors.

When a user enters a query:

1. The query is converted to a vector.
2. The vector store compares it with document vectors.
3. The nearest vectors are returned.

That is **semantic search**.

In [6]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded:", EMBEDDING_MODEL)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2


## 8. Test the embedding model directly

Before creating the database, it is useful to see that text is actually converted into vectors.

In [7]:
sample_texts = [
    "Raskolnikov felt guilty after the murder.",
    "His conscience troubled him deeply.",
    "The weather in the city was cold."
]

vectors = embeddings.embed_documents(sample_texts)

for text, vector in zip(sample_texts, vectors):
    print("TEXT:", text)
    print("VECTOR LENGTH:", len(vector))
    print("FIRST 10 VALUES:", vector[:10])
    print("-" * 80)

TEXT: Raskolnikov felt guilty after the murder.
VECTOR LENGTH: 384
FIRST 10 VALUES: [0.003958616405725479, 0.034782201051712036, -0.09098435938358307, 0.01848265714943409, 0.12135439366102219, -0.04685692489147186, 0.06424116343259811, 0.05483453720808029, 0.014278645627200603, 0.018630245700478554]
--------------------------------------------------------------------------------
TEXT: His conscience troubled him deeply.
VECTOR LENGTH: 384
FIRST 10 VALUES: [0.11375673860311508, 0.1061217412352562, -0.005900821648538113, 0.06502364575862885, 0.0047103953547775745, 0.006490121595561504, 0.0867704302072525, 0.03947056829929352, -0.056933365762233734, -0.07680995762348175]
--------------------------------------------------------------------------------
TEXT: The weather in the city was cold.
VECTOR LENGTH: 384
FIRST 10 VALUES: [0.026603303849697113, 0.09953483939170837, 0.0991179421544075, 0.14728261530399323, 0.04573040083050728, -0.006316698156297207, -0.008580436930060387, -0.00879141595

## 9. Create the Chroma vector store

Now we perform the central operation:

**Document chunks + embeddings → Chroma vector store**

Chroma stores the text chunks together with their vector representations.

The first time this runs, it may take a while because the model must generate an embedding for every chunk.

In [8]:
CHROMA_DIR = "./chroma_crime_punishment"

# Remove an old experiment database so this notebook starts clean.
if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="crime_and_punishment",
    persist_directory=CHROMA_DIR
)

print("Vector store created successfully.")
print("Indexed chunks:", len(chunks))

Vector store created successfully.
Indexed chunks: 2856


## 10. Perform your first semantic search

This is the most important cell in the notebook.

We ask:

> Why did Raskolnikov feel guilty?

The system does **not** need the exact words to occur in the same order. It searches according to semantic similarity between the query embedding and the document embeddings.

In [9]:
query = "Why did Raskolnikov feel guilty?"

results = vectorstore.similarity_search(query, k=5)

for i, doc in enumerate(results, start=1):
    print("=" * 100)
    print(f"RESULT {i}")
    print("PAGE:", doc.metadata.get("page"))
    print(doc.page_content[:1500])

RESULT 1
PAGE: 755
his pride had been stung to the quick. It was wounded pride 
that made him ill. Oh, how happy he would have been if 
he could have blamed himself! He could have borne any -
thing then, even shame and disgrace. But he judged himself 
severely, and his exasperated conscience found no par -
ticularly terrible fault in his past, except a simple blunder 
which might happen to anyone. He was ashamed just be -
cause he, Raskolnikov, had so hopelessly, stupidly come to
RESULT 2
PAGE: 745
Raskolnikov was not quite like an ordinary murderer and 
robber, but that there was another element in the case.
To the intense annoyance of those who maintained this 
opinion, the criminal scarcely attempted to defend himself. 
To the decisive question as to what motive impelled him
RESULT 3
PAGE: 100
101Free eBooks at Planet eBook.com
‘But I think, if you would not do it yourself, there’s no 
justice about it…. Let us have another game.’
Raskolnikov was violently agitated. Of course, it wa

## 11. Understand what happened internally

For the query:

**"Why did Raskolnikov feel guilty?"**

the system performs:

```text
User query
    ↓
Query embedding
    ↓
Compare query vector with stored vectors
    ↓
Rank chunks by similarity
    ↓
Return top-k chunks
```

The `k=5` parameter means:

> Return the five most similar chunks.

This does **not** guarantee that all five are relevant. That is exactly why we need evaluation and experimentation.

In [10]:
def retrieve(query, k=5):
    results = vectorstore.similarity_search(query, k=k)
    return results

query = "What caused Raskolnikov to feel guilt?"

results = retrieve(query, k=5)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} | Page {doc.metadata.get('page')} ---")
    print(doc.page_content[:1000])


--- Result 1 | Page 755 ---
his pride had been stung to the quick. It was wounded pride 
that made him ill. Oh, how happy he would have been if 
he could have blamed himself! He could have borne any -
thing then, even shame and disgrace. But he judged himself 
severely, and his exasperated conscience found no par -
ticularly terrible fault in his past, except a simple blunder 
which might happen to anyone. He was ashamed just be -
cause he, Raskolnikov, had so hopelessly, stupidly come to

--- Result 2 | Page 745 ---
Raskolnikov was not quite like an ordinary murderer and 
robber, but that there was another element in the case.
To the intense annoyance of those who maintained this 
opinion, the criminal scarcely attempted to defend himself. 
To the decisive question as to what motive impelled him

--- Result 3 | Page 436 ---
to express his joy fully, but he was in a fever of excitement as 
though a ton-weight had fallen off his heart. Now he had the 
right to devote his life to them, 

## 12. Test multiple queries

One query is not enough to evaluate a retrieval system.

We create a small evaluation set covering different aspects of the book.

These are **test queries**, not claims about the exact answers. The purpose is to see whether the retrieval system returns passages that are actually relevant.

In [11]:
test_queries = [
    "Why did Raskolnikov commit the murder?",
    "Why did Raskolnikov feel guilty?",
    "What role does Sonia play in Raskolnikov's life?",
    "What is Raskolnikov's relationship with his mother?",
    "What happens after the murder?",
    "How does Porfiry investigate Raskolnikov?"
]

for query in test_queries:
    print("\n" + "=" * 100)
    print("QUERY:", query)
    results = retrieve(query, k=3)

    for i, doc in enumerate(results, 1):
        print(f"\nResult {i} | Page {doc.metadata.get('page')}")
        print(doc.page_content[:500].replace("\n", " "))


QUERY: Why did Raskolnikov commit the murder?

Result 1 | Page 745
Raskolnikov was not quite like an ordinary murderer and  robber, but that there was another element in the case. To the intense annoyance of those who maintained this  opinion, the criminal scarcely attempted to defend himself.  To the decisive question as to what motive impelled him

Result 2 | Page 747
Crime and Punishment Raskolnikov was at the university he had helped a poor  consumptive fellow student and had spent his last penny  on supporting him for six months, and when this student  died, leaving a decrepit old father whom he had maintained  almost from his thirteenth year, Raskolnikov had got the  old man into a hospital and paid for his funeral when he  died. Raskolnikov’s landlady bore witness, too, that when  they had lived in another house at Five Corners, Raskol -

Result 3 | Page 735
Crime and Punishment had followed him then on his painful way! Raskolnikov at  that moment felt and knew once for a

# 13. Experiment 1 — Different chunk sizes

This is the main experiment suggested by the assignment.

We compare:

- 200 characters
- 500 characters
- 800 characters
- 1000 characters

### What are we looking for?

There is a trade-off:

**Very small chunks**
- More focused
- Less irrelevant information
- But may lose context

**Very large chunks**
- More context
- But may contain unrelated information
- Retrieval can become less precise

We therefore should not simply assume that the largest or smallest chunk is best.

In [12]:
def make_chunks(chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    return splitter.split_documents(pages)

chunk_settings = [
    (200, 20),
    (500, 50),
    (800, 80),
    (1000, 100)
]

chunk_results = []

for size, overlap in chunk_settings:
    test_chunks = make_chunks(size, overlap)
    chunk_results.append({
        "chunk_size": size,
        "overlap": overlap,
        "number_of_chunks": len(test_chunks),
        "avg_chars": round(
            sum(len(d.page_content) for d in test_chunks) / len(test_chunks), 1
        )
    })

chunk_df = pd.DataFrame(chunk_results)
chunk_df

,chunk_size,overlap,number_of_chunks,avg_chars
0,200,20,6920,168.3
1,500,50,2856,412.1
2,800,80,2016,616.3
3,1000,100,1501,810.7


## 14. Build a vector store for each chunk configuration

To make a fair comparison, we create a separate vector store for each chunking configuration.

This is computationally more expensive than the first example because every chunk must be embedded.

For a class assignment, this is useful because it gives us experimental evidence instead of guessing.

In [13]:
experiment_stores = {}

for size, overlap in chunk_settings:
    print(f"Building store: chunk_size={size}, overlap={overlap}")

    experiment_chunks = make_chunks(size, overlap)

    store_dir = f"./chroma_exp_{size}"
    if os.path.exists(store_dir):
        shutil.rmtree(store_dir)

    store = Chroma.from_documents(
        documents=experiment_chunks,
        embedding=embeddings,
        collection_name=f"crime_{size}",
        persist_directory=store_dir
    )

    experiment_stores[size] = store

print("\nAll chunk-size experiments are ready.")

Building store: chunk_size=200, overlap=20
Building store: chunk_size=500, overlap=50
Building store: chunk_size=800, overlap=80
Building store: chunk_size=1000, overlap=100

All chunk-size experiments are ready.


## 15. Compare retrieval results

Now we can run the **same query** against all four configurations.

This is important: the query stays fixed, while the chunking strategy changes.

That makes the comparison more meaningful.

In [14]:
comparison_query = "Why did Raskolnikov feel guilty?"

for size in [200, 500, 800, 1000]:
    print("\n" + "#" * 100)
    print(f"CHUNK SIZE = {size}")

    results = experiment_stores[size].similarity_search(
        comparison_query,
        k=3
    )

    for i, doc in enumerate(results, 1):
        print(f"\nResult {i} | Page {doc.metadata.get('page')}")
        print(doc.page_content[:700].replace("\n", " "))


####################################################################################################
CHUNK SIZE = 200

Result 1 | Page 432
dictive hatred as he felt against Raskolnikov. Him, and him  alone, he blamed for everything. It is noteworthy that as he  went downstairs he still imagined that his case was perhaps

Result 2 | Page 745
Raskolnikov was not quite like an ordinary murderer and  robber, but that there was another element in the case. To the intense annoyance of those who maintained this

Result 3 | Page 755
ticularly terrible fault in his past, except a simple blunder  which might happen to anyone. He was ashamed just be - cause he, Raskolnikov, had so hopelessly, stupidly come to

####################################################################################################
CHUNK SIZE = 500

Result 1 | Page 755
his pride had been stung to the quick. It was wounded pride  that made him ill. Oh, how happy he would have been if  he could have blamed himself! He c

## 16. Experiment 2 — Different values of k

`k` controls how many passages are returned.

Try:

- `k=1`
- `k=3`
- `k=5`
- `k=10`

### Interpretation

`k=1` gives only the single best result.

That may be too little context.

`k=10` gives much more information, but some passages may be less relevant.

The goal is not "maximum k". The goal is **useful retrieval**.

In [15]:
k_values = [1, 3, 5, 10]

k_query = "Why did Raskolnikov commit the murder?"

for k in k_values:
    results = vectorstore.similarity_search(k_query, k=k)

    print("\n" + "=" * 100)
    print(f"k = {k}")

    for i, doc in enumerate(results, 1):
        print(f"Result {i} | Page {doc.metadata.get('page')} | "
              f"{doc.page_content[:250].replace(chr(10), ' ')}")


k = 1
Result 1 | Page 745 | Raskolnikov was not quite like an ordinary murderer and  robber, but that there was another element in the case. To the intense annoyance of those who maintained this  opinion, the criminal scarcely attempted to defend himself.  To the decisive quest

k = 3
Result 1 | Page 745 | Raskolnikov was not quite like an ordinary murderer and  robber, but that there was another element in the case. To the intense annoyance of those who maintained this  opinion, the criminal scarcely attempted to defend himself.  To the decisive quest
Result 2 | Page 747 | Crime and Punishment Raskolnikov was at the university he had helped a poor  consumptive fellow student and had spent his last penny  on supporting him for six months, and when this student  died, leaving a decrepit old father whom he had maintain
Result 3 | Page 735 | Crime and Punishment had followed him then on his painful way! Raskolnikov at  that moment felt and knew once for all that Sonia was with  him

# 17. A simple relevance evaluation

A vector search system needs an evaluation method.

For a classroom experiment, we can manually judge each retrieved passage:

- **1 = relevant**
- **0 = not relevant**

For example, if the query is:

> Why did Raskolnikov feel guilty?

and the top three passages are:

```text
Passage 1 → directly discusses his guilt       = 1
Passage 2 → discusses his psychological state  = 1
Passage 3 → describes unrelated weather        = 0
```

Then:

**Precision@3 = 2 / 3 = 0.67**

This is a simple but understandable way to demonstrate retrieval quality.

In [16]:
# Example manual relevance scores.
# Replace these values after reading the actual retrieved passages.

example_scores = {
    "Why did Raskolnikov feel guilty?": [1, 1, 0],
    "Why did Raskolnikov commit the murder?": [1, 1, 1],
}

for query, scores in example_scores.items():
    precision = sum(scores) / len(scores)
    print(f"{query}")
    print(f"Precision@{len(scores)} = {precision:.2f}")

Why did Raskolnikov feel guilty?
Precision@3 = 0.67
Why did Raskolnikov commit the murder?
Precision@3 = 1.00


## 18. Create a reusable evaluation function

Instead of manually repeating the calculation, we create a function.

You provide the query and a list such as:

```python
[1, 1, 0, 1, 0]
```

The function calculates Precision@k.

In [17]:
def precision_at_k(relevance_scores):
    if not relevance_scores:
        return 0.0
    return sum(relevance_scores) / len(relevance_scores)

print("Example Precision@5:",
      precision_at_k([1, 1, 0, 1, 0]))

Example Precision@5: 0.6


# 19. Optional: compare with FAISS

The assignment names several vector-store technologies. Cloud/server-based systems such as Pinecone, Weaviate, and Milvus may require additional configuration.

For a local classroom comparison, FAISS is another useful vector-search implementation.

Install it with:

```python
!pip install -q faiss-cpu
```

Then LangChain can build a local FAISS index.

This section is **optional**. The main assignment is already completed with ChromaDB and retrieval experiments.

In [18]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.4 MB/s eta 0:00:00


In [19]:
from langchain_community.vectorstores import FAISS

faiss_store = FAISS.from_documents(
    chunks,
    embeddings
)

faiss_results = faiss_store.similarity_search(
    "Why did Raskolnikov feel guilty?",
    k=5
)

for i, doc in enumerate(faiss_results, 1):
    print(f"\nResult {i} | Page {doc.metadata.get('page')}")
    print(doc.page_content[:700])


Result 1 | Page 755
his pride had been stung to the quick. It was wounded pride 
that made him ill. Oh, how happy he would have been if 
he could have blamed himself! He could have borne any -
thing then, even shame and disgrace. But he judged himself 
severely, and his exasperated conscience found no par -
ticularly terrible fault in his past, except a simple blunder 
which might happen to anyone. He was ashamed just be -
cause he, Raskolnikov, had so hopelessly, stupidly come to

Result 2 | Page 745
Raskolnikov was not quite like an ordinary murderer and 
robber, but that there was another element in the case.
To the intense annoyance of those who maintained this 
opinion, the criminal scarcely attempted to defend himself. 
To the decisive question as to what motive impelled him

Result 3 | Page 100
101Free eBooks at Planet eBook.com
‘But I think, if you would not do it yourself, there’s no 
justice about it…. Let us have another game.’
Raskolnikov was violently agitated. Of course,

## 20. ChromaDB vs FAISS — what are we comparing?

For this notebook, the important conceptual distinction is:

### ChromaDB
A vector database/store designed for storing embeddings and associated documents, with persistence and metadata support.

### FAISS
A library focused on efficient similarity search over vectors. It is especially useful for local indexing/search experiments.

For this assignment, **retrieval quality is more important than simply naming the database**.

The key question is:

> Which configuration returns the most appropriate passages for our queries?



# 21. Build a compact experiment table

A good assignment report should contain a table showing what was tested.

The following table records the experimental variables. You should add your manually judged relevance scores after examining the results.

In [20]:
experiment_table = pd.DataFrame([
    {"Experiment": "Baseline", "Chunk Size": 500, "Overlap": 50, "k": 3, "Vector Store": "Chroma", "Precision@k": ""},
    {"Experiment": "Small chunks", "Chunk Size": 200, "Overlap": 20, "k": 3, "Vector Store": "Chroma", "Precision@k": ""},
    {"Experiment": "Medium chunks", "Chunk Size": 500, "Overlap": 50, "k": 3, "Vector Store": "Chroma", "Precision@k": ""},
    {"Experiment": "Large chunks", "Chunk Size": 800, "Overlap": 80, "k": 3, "Vector Store": "Chroma", "Precision@k": ""},
    {"Experiment": "Very large chunks", "Chunk Size": 1000, "Overlap": 100, "k": 3, "Vector Store": "Chroma", "Precision@k": ""},
    {"Experiment": "More results", "Chunk Size": 500, "Overlap": 50, "k": 5, "Vector Store": "Chroma", "Precision@k": ""},
    {"Experiment": "FAISS comparison", "Chunk Size": 500, "Overlap": 50, "k": 3, "Vector Store": "FAISS", "Precision@k": ""}
])

experiment_table

,Experiment,Chunk Size,Overlap,k,Vector Store,Precision@k
0,Baseline,500,50,3,Chroma,
1,Small chunks,200,20,3,Chroma,
2,Medium chunks,500,50,3,Chroma,
3,Large chunks,800,80,3,Chroma,
4,Very large chunks,1000,100,3,Chroma,
5,More results,500,50,5,Chroma,
6,FAISS comparison,500,50,3,FAISS,


# 22. What should you conclude?

Do **not** invent a winning configuration before running the experiments.

Your conclusion should be based on the passages you actually observe.

A good conclusion structure is:

1. State which chunk sizes were tested.
2. State which overlap values were tested.
3. State which values of `k` were tested.
4. Explain which configuration produced the most relevant passages.
5. Explain the trade-off between context and precision.
6. Mention the embedding model used.
7. Mention the vector store used.
8. Report your evaluation measure, such as Precision@k.

### Example conclusion format

> We developed a semantic retrieval system for Crime and Punishment using the Hugging Face `all-MiniLM-L6-v2` embedding model and ChromaDB. The book was divided into overlapping chunks and indexed as vectors. We experimented with different chunk sizes, overlap values, and retrieval values of k. Smaller chunks produced focused passages but sometimes lacked context, while larger chunks preserved more context but could include irrelevant material. Based on manual relevance evaluation, the configuration with the highest retrieval precision was selected as the final configuration.

**Important:** Replace "the configuration with the highest retrieval precision" with your actual result after performing the evaluation.

# 23. Key concepts you should be able to explain in class

### What is an embedding?
A numerical representation of text that captures semantic information.

### What is a vector?
A list of numerical values representing an item in an embedding space.

### What is a vector store?
A system that stores vectors and allows similarity-based retrieval.

### What is semantic search?
Searching based on meaning rather than only exact keyword matching.

### Why do we chunk documents?
Because meaningful retrieval normally requires smaller passages rather than one vector for an entire large document.

### What is chunk overlap?
Shared text between neighboring chunks, used to reduce loss of context at chunk boundaries.

### What is `k`?
The number of top results returned by a similarity search.

### Why use a Hugging Face model?
The assignment requires an open-source embedding model from Hugging Face.

### What is the objective of this assignment?
To build **effective retrieval**, meaning that queries should return the most appropriate parts of the book.

# 24. Final checklist

Before submitting the assignment, make sure you have:

- [ ] Downloaded and loaded Crime and Punishment
- [ ] Split the document into chunks
- [ ] Used an open-source Hugging Face embedding model
- [ ] Created embeddings
- [ ] Created a Chroma vector store
- [ ] Performed semantic searches
- [ ] Tested multiple queries
- [ ] Tested different chunk sizes
- [ ] Tested different overlap values
- [ ] Tested different values of `k`
- [ ] Compared retrieval results
- [ ] Manually evaluated relevance
- [ ] Recorded Precision@k
- [ ] Selected a final configuration based on evidence
- [ ] Written a short conclusion

## Final takeaway

The central idea is:

**Good Retrieval = Good Chunking + Good Embeddings + Good Similarity Search + Evidence-based Evaluation**

The purpose is not merely to create a vector database. The purpose is to make sure that when someone asks a question, the system retrieves the **right passages**.